# BladeDelta quick start

Evidence-bounded 3D damage estimation: given a healthy CAD model and two photographs, report a dent (location and depth) only when the images support it better than camera error, lighting or a stain.

**Runtime:** select *Runtime → Change runtime type → GPU* before running. One case takes about 1–2 minutes on a T4.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code and install
nvdiffrast compiles its CUDA kernels on first use (a minute or two).

In [ ]:
!git clone -q https://github.com/afeefaaazam03/3D-Jetblade.git
%cd 3D-Jetblade/code
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation
!pip install -q -e .

## 3. Load an example
Each example holds the two input photographs (views v004 and v005), healthy CAD renders from the same cameras, the ground truth, and precomputed outputs for comparison.

In [ ]:
from bladedelta.utils import list_examples, load_example
print(list_examples())
ex = load_example('01_dent_found')
print('truth:', ex['truth'])

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, t in zip(axs, ex['observed'] + ex['healthy'], ['photo v004', 'photo v005', 'healthy v004', 'healthy v005']):
    ax.imshow(im); ax.set_title(t); ax.axis('off')

## 4. Run the method (v1.1)

In [ ]:
from bladedelta import analyze
res = analyze(ex['observed'], ex['healthy'], source_id=ex['truth']['source'], version='v1.1')
print(res.decision, f'| depth {res.depth_mm:.3f} mm | evidence {res.fit_gain:.2f} vs threshold {res.threshold:.2f}')
print('precomputed:', ex['precomputed']['v1.1']['decision'], round(ex['precomputed']['v1.1']['fit_gain'], 2))

## 5. Visualise the reported dent
Blue: vertices the method moved, projected into camera v005.

In [ ]:
import numpy as np
from bladedelta.cameras import camera
from bladedelta.geometry import surface_context
ctx = surface_context(ex['truth']['source'])
sel = np.linalg.norm(res.displacement, axis=1) > 2e-5
c = camera('v005'); w = c['w2c_cv']
X = ctx['v'][sel] @ w[:3, :3].T + w[:3, 3]; uv = X @ c['K'].T; uv = uv[:, :2] / uv[:, 2:]
plt.figure(figsize=(6, 6)); plt.imshow(ex['observed'][1]); plt.scatter(uv[:, 0], uv[:, 1], s=3, c='tab:blue'); plt.axis('off')
plt.title(f'{res.decision}: {res.depth_mm:.2f} mm');

## 6. Try the other cases
A stain, a hidden dent, a healthy blade and a moved camera should all come back **not supported**.

In [ ]:
for name in ['02_hidden_dent', '03_stain_only', '04_healthy', '05_camera_error']:
    e = load_example(name)
    r = analyze(e['observed'], e['healthy'], source_id=e['truth']['source'], version='v1.1')
    print(f'{name:16s} {r.decision:15s} evidence {r.fit_gain:6.2f}  threshold {r.threshold:.2f}')

## Notes
- Inputs must come from the BladeDelta camera rig (`bladedelta/cameras.py`).
- `version='v1'` runs the frozen model used for the paper's main sealed test; `v1.1` adds the reference correction that transfers to new blade designs.
- Paper: `paper/BladeDelta.pdf`. Results: https://afeefaaazam03.github.io/3D-Jetblade/results.html